In [27]:
import os

import torch 

import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
import scLEMBAS.utilities as utils
from scLEMBAS.predict import get_prediction

sys.path.insert(1, '../.')
from McCauley_utils import all_data

sys.path.insert(1, '../../.') 
from notebook_utils import get_split



In [2]:
seed = 888
data_path = '/home/hmbaghda/orcd/pool/scLEMBAS/analysis'
author = 'McCauley'

n_cores = 30
os.environ["OMP_NUM_THREADS"] = str(1)
os.environ["MKL_NUM_THREADS"] = str(1)
os.environ["OPENBLAS_NUM_THREADS"] = str(1)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(1)
os.environ["NUMEXPR_NUM_THREADS"] = str(1)

In [3]:
(sn_ppis, tf_adata, adata, expr, source_label, target_label, weight_label, 
 stimulation_label, inhibition_label, cat_col, pert_col, ctrl_pert) = all_data


In [128]:
ensemble_idx = 0
fold = 0

prediction_type = 'train_counterfactual'
# 'test_counterfactual': test conditions (cell type A ^ perturbation I) predicted from respective controls (cell type A ^ unperturbed)
# 'train_nocounterfactual': all train conditions predicted with no counterfactual ((cell type A ^ perturbation I) predicted from (cell type A ^ perturbation I))
# 'train_counterfactual': for perturbed (non-control) conditions in the training set, predicts from respective controls

condition_subset = ['Basal^TGFB1']



In [138]:
from typing import Literal, Optional

def run_forward_pass(
    fold: int, 
    ensemble_idx: int, 
    prediction_type: Literal['test_counterfactual', 'train_nocounterfactual', 'train_counterfactual'] = 'train_counterfactual',
    condition_subset: Optional[list] = None
):
    """Runs a prediction (forward pass) for an ensemble trained model.

    Parameters
    ----------
    fold : int
        training split (5 splits, so can be any value between 0 and 4)
    ensemble_idx : int
        which model in ensemble from fold (5 per fold, so can be any value between 0 and 4)
    prediction_type : Literal['test_counterfactual', 'train_nocounterfactual', 'train_counterfactual']
        how to run the forward pass
            - 'test_counterfactual': test conditions (cell type A ^ perturbation I) predicted from respective controls (cell type A ^ unperturbed)
            - 'train_nocounterfactual': all train conditions predicted with no counterfactual ((cell type A ^ perturbation I) predicted from (cell type A ^ perturbation I))
            - 'train_counterfactual': for perturbed (non-control) conditions in the training set, predicts from respective controls
    condition_subset : Optional[list], optional
        whether to run forward pass only on a subset of conditions, by default None
        * note, will always include corresponding controls
        
    Returns
    ----------
    tf_adata_predicted : 
        predicted values as an AnnData object
        `.obs` specifies the predicted coundition and the corresponding counterfactual
    mod : 
        the trained model from that fold/ensemble, after running the forward pass
    """
    

    ##### LOAD DATA #####
    if ensemble_idx <= 3:
        fn_trainer =  os.path.join(data_path, 'processed', '{}_fold{}trainer_actual_ensemble{}.pickle'.format(author, fold, ensemble_idx))
    else:
        fn_trainer = os.path.join(data_path, 'processed', '{}_fold{}trainer_actual.pickle'.format(author, fold))


    trainer = io.read_pickled_object(fn_trainer)
    mod = trainer.mod

    split = get_split(fold = fold, author = author)

    train_barcodes = split['train_barcodes']
    test_barcodes = split['test_barcodes']

    assert trainer.X_train.index.tolist() == train_barcodes, 'Train barcodes mismatch'
    assert trainer.X_test.index.tolist() == test_barcodes, 'Test barcodes mismatch'


    if condition_subset is not None:
        ctrl_conds = [cond.split('^')[0] + '^' + ctrl_pert for cond in condition_subset]
        condition_subset += ctrl_conds
        condition_subset = sorted(set(condition_subset))

        condition_mask = tf_adata.obs.condition.isin(condition_subset)
        train_cond_mask = tf_adata[train_barcodes, :].obs.condition.isin(condition_subset)
        test_cond_mask = tf_adata[test_barcodes, :].obs.condition.isin(condition_subset)

        train_barcodes = train_cond_mask[train_cond_mask].index.tolist()
        test_barcodes = test_cond_mask[test_cond_mask].index.tolist()


    # #### RUN Forward PASS ####
    # Note for Nikos: can adapt the 

    if prediction_type == 'test_counterfactual':
        tf_adata_predicted = get_prediction(
            mod = mod,
            train_cells = train_barcodes,
            test_cells = test_barcodes, 
            tf_adata = tf_adata,
            cat_col = cat_col,
            pert_col = pert_col,
            ctrl_pert = ctrl_pert, 
            counterfactual = 'perturbation', # counterfactual from tests
            cat_counterfactual_map = None,
            remove_type = 'none',
            return_bias = False, 
            max_cells = int(5e3), 
            return_full = False, 
            stim_label_map = None, # special use case for Kang
        )
    elif prediction_type == 'train_nocounterfactual':
        tf_adata_predicted = get_prediction(
            mod = mod,
            train_cells = train_barcodes,
            test_cells = [], 
            tf_adata = tf_adata,
            cat_col = cat_col,
            pert_col = pert_col,
            ctrl_pert = ctrl_pert, 
            counterfactual = None,
            cat_counterfactual_map = None,
            remove_type = 'none',
            return_bias = False, 
            max_cells = int(5e3), 
            return_full = False,
            stim_label_map = None, # special use case for Kang
        )
    elif prediction_type == 'train_counterfactual':
        ctrl_mask = tf_adata[train_barcodes, :].obs[pert_col] == ctrl_pert
        train_barcodes_ctrl = pd.Series(train_barcodes)[ctrl_mask.tolist()]
        train_barcodes_pert = pd.Series(train_barcodes)[(~ctrl_mask).tolist()]


        tf_adata_predicted = get_prediction(
            mod = mod,
            train_cells = train_barcodes_ctrl,
            test_cells = train_barcodes_pert, 
            tf_adata = tf_adata,
            cat_col = cat_col,
            pert_col = pert_col,
            ctrl_pert = ctrl_pert, 
            counterfactual = 'perturbation',
            cat_counterfactual_map = None,
            remove_type = 'none',
            return_bias = False, 
            max_cells = int(5e3), 
            return_full = False,
            stim_label_map = None, # special use case for Kang
        )
    return tf_adata_predicted, mod

In [139]:
# example usage
fold = 0
ensemble_idx = 0

tf_adata_predicted, mod = run_forward_pass(
    fold = fold, 
    ensemble_idx = ensemble_idx, 
    prediction_type = 'train_counterfactual', 
    condition_subset = ['Basal^TGFB1'], 
)
signaling_weights = mod.signaling_network.weights

Set up inputs for prediction


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00,  1.84it/s]


Get the predictions


/orcd/pool/005/hmbaghda/miniforge3/envs/scLEMBAS/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
